# Usage of GB_code (v2)

This notebook demonstrates the full workflow for constructing grain boundaries with GB_code:

1. List CSL boundaries for a rotation axis
2. Select a sigma value and inspect GB characteristics
3. Browse GB planes by type, atom count, or proximity to symmetric tilts
4. Build the bicrystal and export to pymatgen, ASE, LAMMPS, or VASP

In [ ]:
import numpy as np
from numpy.linalg import norm
from math import atan2, asin, degrees
import pandas as pd
import matplotlib.pyplot as plt

import gb_code.csl_generator as csl
from gb_code import GB_character

## 1. List CSL boundaries for a rotation axis

In [ ]:
axis = np.array([1, 1, 1])

# List all Sigma boundaries with Sigma < 50
csl.print_list(axis, 50)

## 2. Select a Sigma and inspect GB characteristics

In [ ]:
sigma = 19

theta, m, n = csl.get_theta_m_n_list(axis, sigma)[0]
R = csl.rot(axis, theta)

# Minimal CSL cells (used internally for plane and orthogonal cell generation)
M1, M2 = csl.Create_minimal_cell_Method_1(sigma, axis, R)

print(f'Angle: {degrees(theta):.2f} deg')
print(f'Sigma: {sigma}')
print(f'Minimal cells:\n{M1}\n{M2}')

## 3. Browse GB planes

Generate a list of possible GB planes. Increase `lim` for higher-index planes.

In [ ]:
lim = 3
V1, V2, M, Gb = csl.Create_Possible_GB_Plane_List(axis, m, n, lim)

df = pd.DataFrame({'GB1': list(V1), 'GB2': list(V2), 'Type': Gb})
df.head()

### Filter by GB type

Available types: `Symmetric Tilt`, `Tilt`, `Twist`, `Mixed`.

In [ ]:
df[df['Type'] == 'Tilt'].head()

### Filter by number of atoms in the orthogonal cell

Useful for DFT calculations that require smaller cells. This computes all orthogonal cells, so it may take a moment for large plane lists.

In [ ]:
basis = 'fcc'

num_atoms = [csl.Find_Orthogonal_cell(basis, axis, m, n, V1[i])[2]
             for i in range(len(V1))]

df['NumAtoms'] = num_atoms

max_atoms = 300
df[df['NumAtoms'] < max_atoms]

### Filter by proximity to symmetric tilt boundaries

Find GB planes within a given angular distance from any symmetric tilt in this system. This approach was used to create vicinal grain boundaries in [Hadian et al., Phys. Rev. Materials 2, 043601 (2018)](https://journals.aps.org/prmaterials/abstract/10.1103/PhysRevMaterials.2.043601).

In [ ]:
symm_tilts = [V1[i] for i in range(len(V1)) if Gb[i] == 'Symmetric Tilt']

delta = 6  # degrees
min_angles = [min(csl.angv(V1[i], st) for st in symm_tilts)
              for i in range(len(V1))]

df['AngleToSymmTilt'] = min_angles
df[df['AngleToSymmTilt'] < delta]

## 4. Build the bicrystal (v2 API)

Pick a GB plane from the lists above and build the grain boundary structure.

In [ ]:
GB_plane = [-2, -3, 5]
LatP = 4.0

# Check the tilt/twist decomposition
csl.Tilt_Twist_comp(GB_plane, axis, m, n)

In [ ]:
gb = GB_character()
gb.ParseGB(axis, basis, LatP, m, n, GB_plane)
gb.CSL_Bicrystal_Atom_generator()

# Build with overlap removal from grain 1
gb.build(overlap=0.3, whichG='g1', dim=[3, 3, 2])
print(gb)

### Export to pymatgen or ASE

In [ ]:
# pymatgen Structure
struct = gb.to_pymatgen(element='Al')
print(f'pymatgen Structure: {struct.num_sites} sites')
print(struct.lattice)

# ASE Atoms
atoms = gb.to_ase(element='Al')
print(f'\nASE Atoms: {len(atoms)} atoms')
print(f'Cell:\n{atoms.cell[:]}')

### New in v2: `gb_normal` and `min_inplane_dist`

- **`gb_normal`**: Choose which axis (`'x'`, `'y'`, or `'z'`) the GB plane normal aligns with (default: `'x'`).
- **`min_inplane_dist`**: Specify a minimum in-plane cell dimension in Angstroms. The code auto-computes the required supercell replication.

In [ ]:
gb2 = GB_character()
gb2.ParseGB(axis, basis, LatP, m, n, GB_plane)
gb2.CSL_Bicrystal_Atom_generator()

# GB normal along z, in-plane dimensions at least 10 Angstroms
gb2.build(overlap=0.3, whichG='g1', min_inplane_dist=10.0, gb_normal='z')

struct_z = gb2.to_pymatgen(element='Al')
print(f'Lattice (gb_normal=z):\n{struct_z.lattice}')
print(f'\nGrain info: {gb2.get_grain_info()}')

### Write to LAMMPS / VASP files

In [ ]:
# gb.write_lammps('my_gb.lammps')
# gb.write_vasp('POSCAR_my_gb')
print('Uncomment the lines above to write output files.')

## 5. Visualise the bicrystal

In [ ]:
def plot_gb(gb_obj, view_dir=(0, 1, 0)):
    """3D scatter plot of the two grains."""
    X, Y = gb_obj.atoms1, gb_obj.atoms2
    fig = plt.figure(figsize=(8, 8))
    ax = fig.add_subplot(111, projection='3d')
    ax.scatter(X[:, 0], X[:, 1], X[:, 2], s=20, c='gold',
              edgecolors='none', alpha=0.5, label='Grain 1')
    ax.scatter(Y[:, 0], Y[:, 1], Y[:, 2], s=20, c='steelblue',
              edgecolors='none', alpha=0.5, label='Grain 2')
    ax.scatter(0, 0, 0, s=200, c='red', marker='s', label='Origin')
    ax.set_proj_type('ortho')
    ax.grid(False)
    az = degrees(atan2(view_dir[1], view_dir[0]))
    el = degrees(asin(view_dir[2] / norm(view_dir)))
    ax.view_init(azim=az, elev=el)
    ax.set_xlabel('X'); ax.set_ylabel('Y'); ax.set_zlabel('Z')
    ax.legend()
    plt.tight_layout()
    return ax

plot_gb(gb, view_dir=(0, 1, 0))
plt.show()